In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [2]:
# Load the Kaggle competition files.
# Keeping these paths centralized makes it easy to rerun in Kaggle without touching later cells.
DATA_DIR = "/kaggle/input/competitions/playground-series-s6e5"

df = pd.read_csv(f"{DATA_DIR}/train.csv")
TEST_PATH = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import time


# Drop id
df = df.drop(columns=['id'])

# Prepare categorical columns
cat_cols = ['Driver', 'Compound', 'Race']
for col in cat_cols:
    df[col] = df[col].astype('category')

X = df.drop(columns=['PitNextLap'])
y = df['PitNextLap']

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")

# Initialize and train XGBoost
print("Training XGBoost baseline...")
start_time = time.time()
clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    enable_categorical=True,
    tree_method='hist',
    n_jobs=-1
)

clf.fit(X_train, y_train)
elapsed = time.time() - start_time
print(f"Training completed in {elapsed:.2f} seconds.")

# Evaluate
y_train_pred = clf.predict_proba(X_train)[:, 1]
y_val_pred = clf.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(y_train, y_train_pred)
val_auc = roc_auc_score(y_val, y_val_pred)

print(f"Train ROC-AUC: {train_auc:.5f}")
print(f"Validation ROC-AUC: {val_auc:.5f}")

X_train shape: (351312, 14), X_val shape: (87828, 14)
Training XGBoost baseline...
Training completed in 8.80 seconds.
Train ROC-AUC: 0.97477
Validation ROC-AUC: 0.93986


In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
import time

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Data loaded. Train shape:", train_df.shape, "Test shape:", test_df.shape)
# Drop id from features but keep for output
train_ids = train_df['id']
test_ids = test_df['id']
train_df = train_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])
# Feature engineering
print("Engineering features...")
for df in [train_df, test_df]:
    df['Estimated_Total_Laps'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-8)).replace([np.inf, -np.inf], np.nan)
    df['Estimated_Total_Laps'] = df['Estimated_Total_Laps'].fillna(70).round().clip(10, 100)
    df['Remaining_Laps'] = df['Estimated_Total_Laps'] - df['LapNumber']
    df['Degradation_Rate'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-8)
    df['TyreLife_x_Degradation'] = df['TyreLife'] * df['Cumulative_Degradation']
# Median pit tyre life
pit_stops = train_df[train_df['PitNextLap'] == 1]
median_pit = pit_stops.groupby(['Race', 'Compound', 'Stint'])['TyreLife'].median().reset_index()
median_pit.rename(columns={'TyreLife': 'MedianPitTyreLife'}, inplace=True)
global_median = pit_stops['TyreLife'].median()
comp_median = pit_stops.groupby('Compound')['TyreLife'].median().to_dict()
train_df = train_df.merge(median_pit, on=['Race','Compound','Stint'], how='left')
train_df['MedianPitTyreLife'] = train_df['MedianPitTyreLife'].fillna(train_df['Compound'].map(comp_median)).fillna(global_median)
train_df['TyreLife_to_MedianPitRatio'] = train_df['TyreLife'] / (train_df['MedianPitTyreLife'] + 1e-8)
test_df = test_df.merge(median_pit, on=['Race','Compound','Stint'], how='left')
test_df['MedianPitTyreLife'] = test_df['MedianPitTyreLife'].fillna(test_df['Compound'].map(comp_median)).fillna(global_median)
test_df['TyreLife_to_MedianPitRatio'] = test_df['TyreLife'] / (test_df['MedianPitTyreLife'] + 1e-8)
# Median lap time
median_lap = train_df.groupby(['Race','Compound'])['LapTime (s)'].median().reset_index()
median_lap.rename(columns={'LapTime (s)': 'MedianLapTimeRC'}, inplace=True)
global_lap = train_df['LapTime (s)'].median()
train_df = train_df.merge(median_lap, on=['Race','Compound'], how='left')
train_df['MedianLapTimeRC'] = train_df['MedianLapTimeRC'].fillna(global_lap)
train_df['LapTime_Ratio'] = train_df['LapTime (s)'] / (train_df['MedianLapTimeRC'] + 1e-8)
test_df = test_df.merge(median_lap, on=['Race','Compound'], how='left')
test_df['MedianLapTimeRC'] = test_df['MedianLapTimeRC'].fillna(global_lap)
test_df['LapTime_Ratio'] = test_df['LapTime (s)'] / (test_df['MedianLapTimeRC'] + 1e-8)
# Categorical conversion
cat_cols = ['Driver','Compound','Race']
for col in cat_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')
# Prepare features and target
y_train = train_df['PitNextLap']
X_train = train_df.drop(columns=['PitNextLap'])
X_test = test_df.copy()
print('Feature matrix shapes:', X_train.shape, X_test.shape)
# XGBoost model
clf = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    gamma=0.1,
    min_child_weight=3,
    random_state=42,
    enable_categorical=True,
    tree_method='hist',
    n_jobs=-1,
    eval_metric='auc'
)
print('Training final model on full data...')
start = time.time()
clf.fit(X_train, y_train)
print(f'Training completed in {time.time() - start:.2f} seconds.')
print('Predicting on test set...')
test_pred = clf.predict_proba(X_test)[:,1]
pred_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_pred})


Data loaded. Train shape: (439140, 16) Test shape: (188165, 15)
Engineering features...
Feature matrix shapes: (439140, 22) (188165, 22)
Training final model on full data...
Training completed in 52.35 seconds.
Predicting on test set...


In [6]:
OUTPUT_PATH = "submission.csv"

test_pred = clf.predict_proba(X_test)[:,1]

pred_df = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': test_pred
})

pred_df.to_csv(OUTPUT_PATH, index=False)

print(f'Predictions saved to {OUTPUT_PATH}')

Predictions saved to submission.csv


In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import time

# ==========================================
# 1. LOAD DATA & INITIAL SETUP
# ==========================================
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Data loaded. Train shape:", train_df.shape, "Test shape:", test_df.shape)

train_ids = train_df['id']
test_ids = test_df['id']
y_train_original = train_df['PitNextLap']

# ==========================================
# 2. ADVANCED RACING FEATURE ENGINEERING
# ==========================================
def engineer_features_v2(df, is_train=True, pit_stops_context=None):
    df_out = df.copy()
    
    # Core Ratios
    df_out['Estimated_Total_Laps'] = (df_out['LapNumber'] / (df_out['RaceProgress'] + 1e-8)).replace([np.inf, -np.inf], np.nan)
    df_out['Estimated_Total_Laps'] = df_out['Estimated_Total_Laps'].fillna(70).round().clip(10, 100)
    df_out['Remaining_Laps'] = df_out['Estimated_Total_Laps'] - df_out['LapNumber']
    df_out['Degradation_Rate'] = df_out['Cumulative_Degradation'] / (df_out['TyreLife'] + 1e-8)
    df_out['TyreLife_x_Degradation'] = df_out['TyreLife'] * df_out['Cumulative_Degradation']
    
    # Strict Chronological Sorting for Time-Series Windowing
    df_out = df_out.sort_values(by=['Year', 'Race', 'Driver', 'LapNumber'])
    
    # ADVANCED: Tyre Performance Cliff Indicators (Lags & Deltas)
    for lag in [1, 2, 3]:
        df_out[f'LapTime_Lag_{lag}'] = df_out.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].shift(lag)
        df_out[f'Position_Lag_{lag}'] = df_out.groupby(['Year', 'Race', 'Driver'])['Position'].shift(lag)
        df_out[f'Degradation_Lag_{lag}'] = df_out.groupby(['Year', 'Race', 'Driver'])['Cumulative_Degradation'].shift(lag)

    # Achanak speed drop hona (Current Lap vs Previous Lap)
    df_out['LapTime_Drop_vs_Lag1'] = df_out['LapTime (s)'] - df_out['LapTime_Lag_1']
    df_out['Position_Drop_vs_Lag1'] = df_out['Position'] - df_out['Position_Lag_1']
    
    # Rolling Windows (Trend Analysis)
    df_out['LapTime_RollMean_3'] = df_out.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df_out['Degradation_RollMean_3'] = df_out.groupby(['Year', 'Race', 'Driver'])['Cumulative_Degradation'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    
    # Cliff Delta (Agar current lap time rolling mean say boht bada hai = Tyre khatam)
    df_out['LapTime_vs_RollMean_Delta'] = df_out['LapTime (s)'] - df_out['LapTime_RollMean_3']

    # Historical Pit Windows Context
    if is_train:
        pit_stops = df_out[df_out['PitNextLap'] == 1]
        median_pit = pit_stops.groupby(['Race', 'Compound', 'Stint'])['TyreLife'].median().reset_index()
        median_pit.rename(columns={'TyreLife': 'MedianPitTyreLife'}, inplace=True)
        global_median = pit_stops['TyreLife'].median()
        comp_median = pit_stops.groupby('Compound')['TyreLife'].median().to_dict()
        
        median_lap = df_out.groupby(['Race', 'Compound'])['LapTime (s)'].median().reset_index()
        median_lap.rename(columns={'LapTime (s)': 'MedianLapTimeRC'}, inplace=True)
        global_lap = df_out['LapTime (s)'].median()
        
        pit_context = (median_pit, global_median, comp_median, median_lap, global_lap)
    else:
        median_pit, global_median, comp_median, median_lap, global_lap = pit_stops_context
        
    # Merge Statistics
    df_out = df_out.merge(median_pit, on=['Race', 'Compound', 'Stint'], how='left')
    df_out['MedianPitTyreLife'] = df_out['MedianPitTyreLife'].fillna(df_out['Compound'].map(comp_median)).fillna(global_median)
    df_out['TyreLife_to_MedianPitRatio'] = df_out['TyreLife'] / (df_out['MedianPitTyreLife'] + 1e-8)
    df_out['TyreLife_Remaining_Before_Pit'] = df_out['MedianPitTyreLife'] - df_out['TyreLife']
    
    df_out = df_out.merge(median_lap, on=['Race', 'Compound'], how='left')
    df_out['MedianLapTimeRC'] = df_out['MedianLapTimeRC'].fillna(global_lap)
    df_out['LapTime_Ratio'] = df_out['LapTime (s)'] / (df_out['MedianLapTimeRC'] + 1e-8)
    
    # Clean Shift/Rolling NaNs
    df_out = df_out.fillna(0)
    
    # Categorical Features setup
    cat_cols = ['Driver', 'Compound', 'Race']
    for col in cat_cols:
        df_out[col] = df_out[col].astype('category')
        
    if is_train:
        return df_out, pit_context
    return df_out

print("Engineering Train Features with Version 2 Pro...")
train_feat_df, context = engineer_features_v2(train_df, is_train=True)

print("Engineering Test Features with Version 2 Pro...")
test_feat_df = engineer_features_v2(test_df, is_train=False, pit_stops_context=context)

# Re-align indexing back to Kaggle baseline sequence
train_feat_df = train_feat_df.set_index('id').loc[train_ids].reset_index()
test_feat_df = test_feat_df.set_index('id').loc[test_ids].reset_index()

X_train = train_feat_df.drop(columns=['id', 'PitNextLap'])
y_train = train_feat_df['PitNextLap']
X_test = test_feat_df.drop(columns=['id'])

# CatBoost requirements (string indices or category processing mapping)
cat_features_idx = [X_train.columns.get_loc(col) for col in ['Driver', 'Compound', 'Race']]

print(f"Advanced Features Ready. X_train: {X_train.shape}, X_test: {X_test.shape}")

# ==========================================
# 3. 5-FOLD STRATIFIED TRI-MODEL ENSEMBLE
# ==========================================
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_preds = np.zeros(len(X_test))
lgb_preds = np.zeros(len(X_test))
cat_preds = np.zeros(len(X_test))

oof_xgb = np.zeros(len(X_train))
oof_lgb = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))

print("\nTraining Tri-Model Power Ensemble (XGB + LGBM + CatBoost)...")

for fold, (train_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1} ---")
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Model 1: XGBoost Pro
    model_xgb = xgb.XGBClassifier(
        n_estimators=2000, max_depth=6, learning_rate=0.025, subsample=0.7, colsample_bytree=0.7,
        reg_alpha=5.0, reg_lambda=10.0, min_child_weight=20, random_state=42,
        enable_categorical=True, tree_method='hist', n_jobs=-1, eval_metric='auc', early_stopping_rounds=50
    )
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    # Model 2: LightGBM Pro
    model_lgb = LGBMClassifier(
        n_estimators=2000, max_depth=7, learning_rate=0.025, subsample=0.7, colsample_bytree=0.7,
        reg_alpha=5.0, reg_lambda=10.0, min_child_weight=20, random_state=42, n_jobs=-1, verbose=-1
    )
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='auc', callbacks=[early_stopping(50, verbose=False)])
    
    # Model 3: CatBoost Pro (Handles tabular sequences like magic)
    # Convering categories to string for CatBoost compatibility natively
    X_tr_cb = X_tr.copy()
    X_va_cb = X_va.copy()
    X_test_cb = X_test.copy()
    for col in ['Driver', 'Compound', 'Race']:
        X_tr_cb[col] = X_tr_cb[col].astype(str)
        X_va_cb[col] = X_va_cb[col].astype(str)
        X_test_cb[col] = X_test_cb[col].astype(str)
        
    model_cat = CatBoostClassifier(
        iterations=2000, depth=6, learning_rate=0.03, l2_leaf_reg=10, random_seed=42,
        eval_metric='AUC', early_stopping_rounds=50, task_type='CPU', thread_count=-1, verbose=False
    )
    model_cat.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va), cat_features=cat_features_idx)
    
    # Record Out-of-Fold Scores
    oof_xgb[val_idx] = model_xgb.predict_proba(X_va)[:, 1]
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va)[:, 1]
    oof_cat[val_idx] = model_cat.predict_proba(X_va_cb)[:, 1]
    
    f_xgb = roc_auc_score(y_va, oof_xgb[val_idx])
    f_lgb = roc_auc_score(y_va, oof_lgb[val_idx])
    f_cat = roc_auc_score(y_va, oof_cat[val_idx])
    print(f"Fold {fold+1} -> XGB AUC: {f_xgb:.5f} | LGB AUC: {f_lgb:.5f} | CatBoost AUC: {f_cat:.5f}")
    
    # Accumulate Test Predictions
    xgb_preds += model_xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    lgb_preds += model_lgb.predict_proba(X_test)[:, 1] / folds.n_splits
    cat_preds += model_cat.predict_proba(X_test_cb)[:, 1] / folds.n_splits

print("\n==========================================")
print(f"Final OOF XGB AUC:      {roc_auc_score(y_train, oof_xgb):.5f}")
print(f"Final OOF LGB AUC:      {roc_auc_score(y_train, oof_lgb):.5f}")
print(f"Final OOF CatBoost AUC: {roc_auc_score(y_train, oof_cat):.5f}")

# 3-Way Balanced Blend (Ensemble Strategy)
final_test_preds = (xgb_preds * 0.35) + (lgb_preds * 0.35) + (cat_preds * 0.30)

# ==========================================
# 4. SUBMISSION GENERATION
# ==========================================
OUTPUT_PATH = "submission.csv"
pred_df = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': final_test_preds
})
pred_df.to_csv(OUTPUT_PATH, index=False)
print(f'\n[SUCCESS] Ultimate 0.9650+ push code complete! File saved: {OUTPUT_PATH}')

Data loaded. Train shape: (439140, 16) Test shape: (188165, 15)
Engineering Train Features with Version 2 Pro...
Engineering Test Features with Version 2 Pro...
Advanced Features Ready. X_train: (439140, 37), X_test: (188165, 37)

Training Tri-Model Power Ensemble (XGB + LGBM + CatBoost)...

--- Fold 1 ---
Fold 1 -> XGB AUC: 0.94781 | LGB AUC: 0.94494 | CatBoost AUC: 0.94757

--- Fold 2 ---
Fold 2 -> XGB AUC: 0.94599 | LGB AUC: 0.94297 | CatBoost AUC: 0.94530

--- Fold 3 ---
Fold 3 -> XGB AUC: 0.94703 | LGB AUC: 0.94407 | CatBoost AUC: 0.94681

--- Fold 4 ---
Fold 4 -> XGB AUC: 0.94592 | LGB AUC: 0.94307 | CatBoost AUC: 0.94566

--- Fold 5 ---
Fold 5 -> XGB AUC: 0.94710 | LGB AUC: 0.94446 | CatBoost AUC: 0.94653

Final OOF XGB AUC:      0.94676
Final OOF LGB AUC:      0.94389
Final OOF CatBoost AUC: 0.94637

[SUCCESS] Ultimate 0.9650+ push code complete! File saved: submission.csv
